# Feature_Engineering

In [ ]:
#!pip install nltk
#!pip install langdetect
#!pip install googletrans==4.0.0-rc1
#!pip install stopwords


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import nltk
import plotly.express as px
warnings.filterwarnings("ignore")
%matplotlib inline
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from collections import Counter
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer


In [ ]:
df=pd.read_pickle("df_Cleaned_Outliers_Missing.pkl")

with open("spotify_songs_df_text.pkl", "rb") as f:
    df_text = pickle.load(f)


df.head()

df_text.head()

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
#print(df_text)

In [ ]:
df_text.head()

In [ ]:
df_text.info()

In [ ]:
# WORD CLOUD

In [ ]:
from wordcloud import WordCloud

In [ ]:
# track_name

In [ ]:
if 'track_name' in df_text.columns:
    text = df_text['track_name'].astype(str).str.cat(sep=' ')  # Convert to string and join

    # Generate the word cloud
    wordcloud = WordCloud(width=800, height=400, background_color='black').generate(text)

    # Display the word cloud
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title("Word Cloud from Track Names")
    plt.show()
else:
    print("Error: 'track_name' column not found in df_text.")

In [ ]:
# It is essential to clean the column by removing words that describe the track's type.

In [ ]:
### Extract year and month, converting them explicitly to integers

In [ ]:
df['track_album_release_year'] = df['track_album_release_date'].dt.year.astype('Int64')
df['track_album_release_month'] = df['track_album_release_date'].dt.month.astype('Int64')

# Drop the original column if it's no longer needed
df.drop(columns=['track_album_release_date'], inplace=True)

# Display the updated DataFrame
print(df[['track_album_release_year', 'track_album_release_month']].head())

In [ ]:
#Number of Songs per Language

In [ ]:
DetectorFactory.seed = 0  # Ensure consistent results

# Function to detect language
def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"

# Apply language detection **in bulk** using .apply()
df['language'] = df_text['track_name'].astype(str).map(detect_language)

# Display results
print(df['language'].value_counts())  # See detected languages

In [ ]:
# Count occurrences of each language
language_counts = df['language'].value_counts()

# Plot the results
plt.figure(figsize=(12, 6))
sns.barplot(x=language_counts.index, y=language_counts.values, palette="viridis")

# Customize the plot
plt.xlabel("Language")
plt.ylabel("Number of Songs")
plt.title("Distribution of Songs by Language")
plt.xticks(rotation=45)  # Rotate labels for readability
plt.show()

### Translation of Track Names to English

In [ ]:
#Creating new columns in df_text which contains the translated version of track_name, then merging it with df to transfer the new columns.

In [ ]:
# Download required nltk data
nltk.download('punkt')
nltk.download('stopwords')

# Load the dataset
df_text = pd.read_pickle("spotify_songs_df_text.pkl")

# Initialize stopwords for multiple languages
available_languages = stopwords.fileids()
multi_language_stopwords = set()
for lang in available_languages:
    multi_language_stopwords.update(stopwords.words(lang))

# Custom stopwords (optional)
custom_stopwords = {"remix", "mix", "version", "edit", "feat", "radio", "extended", "the", "and"}
stop_words = multi_language_stopwords | custom_stopwords

# Function to clean, translate, and process text
def clean_and_translate(text):
    if pd.isna(text):  # Handle missing values
        return []
    
    text = text.lower()  # Convert to lowercase
    words = word_tokenize(text)  # Tokenize text
    words = [word for word in words if word.isalpha() and word not in stop_words]  # Remove stopwords & non-alphabetic words
    
    # Translate non-English words to English
    translated_words = []
    for word in words:
        try:
            translated_word = GoogleTranslator(source='auto', target='en').translate(word)
            translated_words.append(translated_word)
        except:
            translated_words.append(word)  # Keep original if translation fails
    
    return translated_words

# Apply processing to track names
df_text['cleaned_words'] = df_text['track_name'].apply(clean_and_translate)

df_text['track_name_english'] = df_text['track_name'].apply(lambda text: ' '.join(clean_and_translate(text)))

# Flatten list and count word frequencies
all_words = [word for words in df_text['cleaned_words'] for word in words]
word_counts = Counter(all_words)

# Get the Top 5 Most Frequent Words
top_5_words = [word for word, _ in word_counts.most_common(5)]

# Create new columns for each of the top 5 words
def check_word_presence(words_list, word):
    return int(word in words_list)

for word in top_5_words:
    df_text[word] = df_text['cleaned_words'].apply(lambda words: check_word_presence(words, word))

# Display the updated DataFrame
print(df_text.head())


In [ ]:
# merge function in pandas to move track_name_english from df_text to df

In [ ]:
df = df.merge(df_text[['id', 'track_name_english']], left_on='track_id', right_on='id', how='left')

# Drop the extra 'id' column after merging (optional)
df.drop(columns=['id'], inplace=True)

In [ ]:
df.info()

In [ ]:
# Top 5 Most Frequent Words in 'track_name_english' Column, first stopwords-> lowercase

In [ ]:
# Download NLTK stopwords if needed
nltk.download("stopwords")

# Get all available stopwords from NLTK
available_languages = stopwords.fileids()

# Collect stopwords for multiple languages
multi_language_stopwords = set()
for lang in available_languages:
    multi_language_stopwords.update(stopwords.words(lang))

# Add custom stopwords (common words in track names)
custom_stopwords = {
    "remix", "mix", "version", "edit", "feat", "radio", 
    "extended", "the", "and"
}

# Combine all stopwords
stop_words = multi_language_stopwords | custom_stopwords


# Tokenize and remove stopwords
df["filtered_words"] = (
    df["track_name_english"]
    .str.lower()
    .str.split()
    .apply(lambda words: [word for word in words if word not in stop_words])
)

# Expand words into separate columns
max_words = df["filtered_words"].apply(len).max()
df_word_columns = df["filtered_words"].apply(lambda x: x + [""] * (max_words - len(x)))  # Pad with empty strings
df_word_columns = pd.DataFrame(df_word_columns.tolist(), columns=[f"word_{i+1}" for i in range(max_words)])

# Concatenate the original DataFrame with new word columns
df = pd.concat([df, df_word_columns], axis=1).drop(columns=["filtered_words"])

# Flatten word lists and count frequency
all_words = df_word_columns.values.flatten()
all_words = [word for word in all_words if word]  # Remove empty strings

word_counts = Counter(all_words)
total_words = sum(word_counts.values())

# Calculate percentage of each word
word_percentages = [(word, count, (count / total_words) * 100) for word, count in word_counts.items()]
word_percentages_df = pd.DataFrame(word_percentages, columns=["Word", "Count", "Percentage"])

# Create an interactive bar chart using Plotly
fig = px.bar(
    word_percentages_df,
    x="Word",
    y="Percentage",
    title="Word Frequency as Percentage of Total Words in Track Names",
    text="Percentage",
    labels={"Word": "Word", "Percentage": "Percentage (%)"},
    color="Percentage",
    color_continuous_scale="Blues"
)

# Show chart
fig.show()

# Display final DataFrame with separate word columns
print(df)


In [ ]:
if 'popular_words_df' in df.columns:
    text = df['popular_words_df'].astype(str).str.cat(sep=' ')  # Convert to string and join

    # Generate the word cloud
    wordcloud = WordCloud(width=800, height=400, background_color='black').generate(text)

    # Display the word cloud
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title("Word Cloud from Track Names")
    plt.show()
else:
    print("Error: 'track_name' column not found in df_text.")

In [ ]:
# artist_type

In [ ]:
import pandas as pd
import re

# Sample artist names dataset
df['track_artist_type'] = df['track_artist']

# Keywords indicating a band
band_keywords = ["band", "orchestra", "quartet", "group", "ensemble", "choir", "collective", "crew"]

# Function to classify artist type
def classify_artist(artist_name):
    words = artist_name.split()

    # Check for known band indicators
    if any(keyword.lower() in artist_name.lower() for keyword in band_keywords):
        return "Band"

    # If the name contains '&' or "and", it's likely a Duo
    if "&" in artist_name or " and " in artist_name.lower():
        return "Duo"

    # If it has 3 words (e.g., "Destiny's Child"), classify as Trio
    if len(words) == 3:
        return "Trio"  


In [ ]:
# track_sentiment

In [ ]:

# Download VADER lexicon (only once)
nltk.download("vader_lexicon")

# Initialize VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Sample DataFrame (Replace this with your dataset)
df = df["track_name_english"]

# Function to compute sentiment score
def get_sentiment(text):
    if pd.isna(text) or text.strip() == "":
        return "neutral"  # If the text is empty or NaN, return neutral
    sentiment_score = sia.polarity_scores(text)["compound"]
    if sentiment_score >= 0.05:
        return "positive"
    elif sentiment_score <= -0.05:
        return "negative"
    else:
        return "neutral"

# Apply sentiment analysis to the 'track_name' column
df["track_sentiment"] = df_text['track_name'].apply(get_sentiment)

# Display results
print(df)


In [ ]:
#find type track by name:  "remix", "acoustic", "live", "mix", "edit", "version"

In [ ]:
# Sample DataFrame
df =  df["track_name_english"]

# Define custom words to find in track names
custom_words = {"remix", "acoustic", "live", "mix", "edit", "version"}

# Function to find custom words in a track name
def find_custom_words(track_name):
    if pd.isna(track_name):
        return None
    words = set(track_name.lower().split())  # Convert to lowercase and split into words
    matched_words = words.intersection(custom_words)  # Find matching words
    return ", ".join(matched_words) if matched_words else None  # Join matches or return None

# Apply the function to detect custom words in track names
df["custom_words_found"] = df["track_name_english"].apply(find_custom_words)

# Display results
print(df)


In [ ]:
# artist_popularity

In [ ]:
# Load the CSV file
tracks_df = pd.read_csv(r"C:\Users\Meital atar\Documents\DATA_SCIENCE\ML_PROJECT_spotify_songs\Kaggel_files\tracks.csv")

# Merge on 'artist_id' and keep the 'popularity' column
df_merge = df.merge(tracks_df[['id', 'popularity']], on='artist_id', how='left')

# Display the first few rows to verify
print(df_merge.head())



In [ ]:
 # explicit (0 = No explicit content, 1 = Explicit content)

In [ ]:
# Load datasets
tracks_df = pd.read_csv(r"C:\Users\Meital atar\Documents\DATA_SCIENCE\ML_PROJECT_spotify_songs\Kaggel_files\tracks.csv")

# Ensure 'track_id' and 'id' columns exist
if 'track_id' in df.columns and 'id' in tracks_df.columns:
    # Merge on 'track_id' and keep the 'explicit' column
    df_merge = df.merge(tracks_df[['id', 'explicit']], left_on='track_id', right_on='id', how='left')

    # Display result
    print(df_merge.head())
else:
    print("Columns 'track_id' or 'id' are missing!")



In [ ]:
df.info()

In [ ]:
df.to_csv('df_after_feature_eng.csv')
df.to_pickle ('df_after_feature_eng.pkl')

In [ ]:
## need to check if the new columns know to select on all track even on track that not in english!!!! very importent